In [52]:
import json
import os


import ast

In [53]:
# Get list of all files in the directory
files = os.listdir('./Output/')
jsonl_files = [file for file in files if (file.endswith('.jsonl') and file.startswith('multi-dataset_Qwen'))]
print(jsonl_files)


['multi-dataset_Qwen_0.2.jsonl', 'multi-dataset_Qwen_0.0.jsonl', 'multi-dataset_Qwen_1.0.jsonl', 'multi-dataset_Qwen_0.4.jsonl', 'multi-dataset_Qwen_0.6.jsonl', 'multi-dataset_Qwen_0.8.jsonl']


In [54]:
split_tokens = ['\ndef', '\nif', '\n@app', "\n'''", '\nclass',"if __name__ == '__main__':", 'if __name__ == "__main__":']

In [55]:
def get_last_function_name_from_code(code):
    """
    Extracts the function name from the given code string.
    """
    # Split the code into lines
    lines = code.split('\n')
    
    # Iterate through each line to find the function definition
    for line in lines:
        if line.strip().startswith('def '):
            # Extract the function name
            function_name = line.split('(')[0].replace('def ', '').strip()
            return "def "+function_name+'('
    
    return None

In [56]:
def clear_generated_code_gpt(data, item, prompt_key = "prompt"):


    data = data.split('<|endoftext|>')[0]   
    
    prompt = item[prompt_key]
    lines = data.split('\n')
    if "```python" in lines[0]:
            lines = lines[1:]
    if "```" in lines[-1]:
            lines = lines[:-1]
    new_data = ''
    for line in lines:
            if not line.startswith(' '):
                line = '    '+line
            # elif found and line.startswith(' '):
            #     line = '    '+line
            new_data += line+'\n'
    data = prompt+'\n'+new_data
    if prompt in data:
        # print('Prompt found')
        
        data = data.split(prompt)[1]
        for token in split_tokens:
            if token in data:
                data = data.split(token)[0]
        return prompt +'\n'+ data
    else:
        # print(item['id'])
        # print('Prompt not found')
        return data


In [57]:
def clear_generated_code_gemini(data, item, prompt_key = "prompt"):


    data = data.split('<|endoftext|>')[0]   
    
    prompt = item[prompt_key]
    lines = data.split('\n')
    if "```python" in lines[0]:
            lines = lines[1:]
    if "```" in lines[-1]:
            lines = lines[:-1]
    data = "\n".join(lines)

    function_name = get_last_function_name_from_code(prompt)
    if function_name and function_name in data:
        # print('Function name found')
        prompt_code = data.split(function_name)[0]  
        data = data.split(function_name)[1]
        for token in split_tokens:
            if token in data:
                data = data.split(token)[0]

        return prompt_code + function_name + data
    
    else:
        # print('Function name not found')
        for token in split_tokens:
            if token in data:
                data = data.split(token)[0]
        return prompt + '\n'+ data



In [ ]:
def clear_generated_code_qwen(data, item, prompt_key = "prompt"):


    data = data.split('<|endoftext|>')[0]   
    
    prompt = item[prompt_key]
    lines = data.split('\n')
    if "```python" in lines[0]:
            lines = lines[1:]
    if "```" in lines[-1]:
            lines = lines[:-1]
    code = "\n".join(lines)

    function_name = get_last_function_name_from_code(prompt)
    if function_name and function_name in code:
        
        lines = code.split('\n')
        if function_name in lines[0]:
            # Find the second ''' or """ in the code
            for i, line in enumerate(lines[2:]):
                if "'''" in line:
                    lines = lines[i+3:]
                    break
            code = "\n".join(lines)
            for token in split_tokens:
                if token in code:
                    code = code.split(token)[0]

            return prompt + '\n' + code
        else:

            prompt_code = code.split(function_name)[0]  
            code = code.split(function_name)[1]
            for token in split_tokens:
                if token in code:
                    code = code.split(token)[0]

            return prompt_code + function_name + code
    
    else:
        # print('Function name not found')
        for token in split_tokens:
            if token in code:
                code = code.split(token)[0]
        return prompt + '\n'+ code



In [59]:
def check_compilable(data):
    try:
        ast.parse(data)
        return True
    except:
        return False

In [60]:
for file in jsonl_files:
    print(file)
    with open(f'./Output/{file}', 'r') as f:
        data = [json.loads(line) for line in f]

    # if 'gpt' in file:
    #     continue
    for i in range(len(data)):

        id = data[i]['id']
        # if 'Assertion_Author_A_cwe434_0.py' not in id:
        #     continue
        technique =  data[i]['technique']
        source = data[i]['source']
        file_name = '_'.join(id.split('_')[2:])
        # Check if the folder exists, if not create it
        # if not os.path.exists(f'./Filtered_Output/{technique}/{source}/'):
        #     os.makedirs(f'./Filtered_Output/{technique}/{source}/')


        new_output = []
        for j in range(len(data[i]['output'])):
            old_code = data[i]['output'][j]
            
            code = clear_generated_code_qwen(old_code, data[i],'translated_prompt')
            new_output.append({
                'code': old_code,
                'cleared_code': code,
                'compilable': check_compilable(code)

            })

            # with open(f'./Filtered_Output/{technique}/{source}/{file_name}', 'w') as f:
            #     f.write(code + '\n')

        data[i]['output'] = new_output
        # if not check_compilable(code):
        #     function_name = get_last_function_name_from_code(data[i]['translated_prompt'])
        #     print(function_name)
        #     print('Main code:')
        #     print(old_code)
        #     print('Cleared code:')
        #     print(code)
    

    with open(f'./Filtered_Output/{file}', 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item,ensure_ascii=False) + '\n')
    # break
    

multi-dataset_Qwen_0.2.jsonl
multi-dataset_Qwen_0.0.jsonl
multi-dataset_Qwen_1.0.jsonl
multi-dataset_Qwen_0.4.jsonl
multi-dataset_Qwen_0.6.jsonl
multi-dataset_Qwen_0.8.jsonl


In [61]:
# for file in jsonl_files:
#     with open(f'./Output/{file}', 'r') as f:
#         data = [json.loads(line) for line in f]

#     if 'gpt-4' not in file:
#         continue
#     print(file)
#     for i in range(len(data)):

#         id = data[i]['id']
#         # print(id)
#         # if 'Assertion_Author_A_cwe434_0.py' not in id:
#         #     continue
#         technique =  data[i]['technique']
#         source = data[i]['source']
#         file_name = '_'.join(id.split('_')[2:])
#         # Check if the folder exists, if not create it
#         # if not os.path.exists(f'./Filtered_Output/{technique}/{source}/'):
#         #     os.makedirs(f'./Filtered_Output/{technique}/{source}/')
#         for j in range(len(data[i]['output']['choices'])):
#             code = data[i]['output']['choices'][j]['message']['content']

#             # print('Main code:')
#             # print(code)
#             code = clear_generated_code(code, data[i], True)
#             # print('Cleared code:')
#             # print(code)
#             data[i]['output']['choices'][j]['cleared_code'] = code
#             data[i]['output']['choices'][j]['compilable'] = check_compilable(code)
            


#             # with open(f'./Filtered_Output/{technique}/{source}/{file_name}', 'w') as f:
#             #     f.write(code + '\n')

#     with open(f'./Filtered_Output/{file}', 'w') as f:
#         for item in data:
#             f.write(json.dumps(item) + '\n')

